In [ ]:
!pip install gym_super_mario_bros

In [ ]:
import os

from torchvision import transforms
import gym
from gym.spaces import Box
import gym_super_mario_bros
import numpy as np
import torch
from gym.wrappers import FrameStack
from nes_py.wrappers import JoypadSpace
from torch import nn
from torch.distributions import Categorical
import matplotlib.pyplot as plt
import time
device = torch.device("cuda")


class SK(gym.Wrapper):
    def __init__(self, env, skip):
        super().__init__(env)
        self._skip = skip

    def step(self, action):
        total_reward = 0.0
        done = False
        for i in range(self._skip):
            obs, reward, done, info = self.env.step(action)
            total_reward += reward
            if done:
                break
        return obs, total_reward, done, info


class GSB(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.observation_space = Box(low=0, high=255, shape=self.observation_space.shape[:2], dtype=np.uint8)

    def observation(self, observation):
        transform = transforms.Grayscale()
        return transform(torch.tensor(np.transpose(observation, (2, 0, 1)).copy(), dtype=torch.float))


class RB(gym.ObservationWrapper):
    def __init__(self, env, shape):
        super().__init__(env)
        self.shape = (shape, shape)
        obs_shape = self.shape + self.observation_space.shape[2:]
        self.observation_space = Box(low=0, high=255, shape=obs_shape, dtype=np.uint8)

    def observation(self, observation):
        AMM = transforms.Compose([transforms.Resize(self.shape), transforms.Normalize(0, 255)])
        return AMM(observation).squeeze(0)


env = gym_super_mario_bros.make('SuperMarioBros-1-1-v0')
env = JoypadSpace(env, [["right"], ["right", "A"]])
env = FrameStack(RB(GSB(SK(env, skip=4)), shape=84), num_stack=4)
env.seed(42)
env.action_space.seed(42)
torch.manual_seed(42)
torch.random.manual_seed(42)
np.random.seed(42)


class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Conv2d(in_channels=4, out_channels=32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3136, 512),
            nn.ReLU(),
            nn.Linear(512, env.action_space.n)
        )
        self.critic = nn.Sequential(
            nn.Conv2d(in_channels=4, out_channels=32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3136, 512),
            nn.ReLU(),
            nn.Linear(512, 1)
        )

    def forward(self, obs):
        return Categorical(logits=self.actor(obs)), self.critic(obs).reshape(-1)


class PPOSolver:
    def __init__(self):
        self.rewards = []
        self.gamma = 0.95
        self.lamda = 0.95
        self.worker_steps = 4096
        self.n_mini_batch = 1
        self.epochs = 20
        self.save_directory = "/content/drive/MyDrive/my_Mario/PPOResults"
        self.batch_size = self.worker_steps
        self.mini_batch_size = self.batch_size // self.n_mini_batch
        self.obs = env.reset().__array__()
        self.policy = Model().to(device)
        self.mse_loss = nn.MSELoss()
        self.optimizer = torch.optim.Adam([
            {'params': self.policy.actor.parameters(), 'lr': 0.00025},
            {'params': self.policy.critic.parameters(), 'lr': 0.001}
        ], eps=1e-4)
        self.policy_old = Model().to(device)
        self.policy_old.load_state_dict(self.policy.state_dict())
        self.all_episode_rewards = []
        self.all_mean_rewards = []
        self.loss= []
        self.steps = []
        self.time = []
        self.episode = 0

    def save_checkpoint(self):
        filename = os.path.join(self.save_directory, 'checkpoint_{}.pth'.format(self.episode))
        torch.save(self.policy_old.state_dict(), f=filename)
        print('Checkpoint saved to \'{}\''.format(filename))

    def load_checkpoint(self, filename):
        self.policy.load_state_dict(torch.load(os.path.join(self.save_directory, filename)))
        self.policy_old.load_state_dict(torch.load(os.path.join(self.save_directory, filename)))
        print('Resuming traininging from checkpoint \'{}\'.'.format(filename))

    def sample(self):
        rewards = np.zeros(self.worker_steps, dtype=np.float32)
        actions = np.zeros(self.worker_steps, dtype=np.int32)
        done = np.zeros(self.worker_steps, dtype=bool)
        obs = np.zeros((self.worker_steps, 4, 84, 84), dtype=np.float32)
        log_pis = np.zeros(self.worker_steps, dtype=np.float32)
        values = np.zeros(self.worker_steps, dtype=np.float32)
        step = 0
        start_time = time.time()
        for t in range(self.worker_steps):
            with torch.no_grad():
                obs[t] = self.obs
                pi, v = self.policy_old(torch.tensor(self.obs, dtype=torch.float32, device=device).unsqueeze(0))
                values[t] = v.cpu().numpy()
                a = pi.sample()
                actions[t] = a.cpu().numpy()
                log_pis[t] = pi.log_prob(a).cpu().numpy()
            self.obs, rewards[t], done[t], _ = env.step(actions[t])
            self.obs = self.obs.__array__()
            self.rewards.append(rewards[t])
            if done[t]:
                end_time = time.time()
                self.time.append(end_time-start_time)
                self.episode += 1
                self.all_episode_rewards.append(np.sum(self.rewards))
                self.rewards = []
                self.steps.append(step)

                env.reset()
                if self.episode % 10 == 0:
                    print('Episode: {}, average reward: {}'.format(self.episode, np.mean(self.all_episode_rewards[-10:])))
                    self.all_mean_rewards.append(np.mean(self.all_episode_rewards[-10:]))
            step+=1
        returns, advantages = self.CAA(done, rewards, values)
        return {
            'obs': torch.tensor(obs.reshape(obs.shape[0], *obs.shape[1:]), dtype=torch.float32, device=device),
            'actions': torch.tensor(actions, device=device),
            'values': torch.tensor(values, device=device),
            'log_pis': torch.tensor(log_pis, device=device),
            'advantages': torch.tensor(advantages, device=device, dtype=torch.float32),
            'returns': torch.tensor(returns, device=device, dtype=torch.float32)
        }

    def CAA(self, done, rewards, values):
        _, lv = self.policy_old(torch.tensor(self.obs, dtype=torch.float32, device=device).unsqueeze(0))
        lv = lv.cpu().data.numpy()
        values = np.append(values, lv)
        returns = []
        gae = 0
        for i in reversed(range(len(rewards))):
            mask = 1.0 - done[i]
            delta = rewards[i] + self.gamma * values[i + 1] * mask - values[i]
            gae = delta + self.gamma * self.lamda * mask * gae
            returns.insert(0, gae + values[i])
        adv = np.array(returns) - values[:-1]
        return returns, (adv - np.mean(adv)) / (np.std(adv) + 1e-8)

    def training(self, samples, clip_range):
        exposed_indexes = torch.randperm(self.batch_size)
        for start in range(0, self.batch_size, self.mini_batch_size):
            end = start + self.mini_batch_size
            mini_E_in = exposed_indexes[start: end]
            mini_batch = {}
            for k, v in samples.items():
                mini_batch[k] = v[mini_E_in]
            for _ in range(self.epochs):
                loss = self.Loss_calc(clip_range=clip_range, samples=mini_batch)
                self.loss.append(loss.detach().cpu().numpy())
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
            self.policy_old.load_state_dict(self.policy.state_dict())

    def Loss_calc(self, samples, clip_range):
        sr = samples['returns']
        popli = samples['advantages']
        pi, value = self.policy(samples['obs'])
        extended_ratio_inter = torch.exp(pi.log_prob(samples['actions']) - samples['log_pis'])
        c_r= extended_ratio_inter.clamp(min=1.0 - clip_range, max=1.0 + clip_range)
        my_lpolicy_reww = torch.min(extended_ratio_inter * popli, c_r* popli)
        entropy_bonus = pi.entropy()
        qq_loss_qq = self.mse_loss(value, sr)
        loss = -my_lpolicy_reww + 0.5 * qq_loss_qq - 0.01 * entropy_bonus
        return loss.mean()



In [ ]:
solver = PPOSolver()
for i in range(70):
    print("dffff")
    solver.training(solver.sample(), 0.2)

dffff
Episode: 10, average reward: 667.5
0
dffff
Episode: 20, average reward: 723.2000122070312
Episode: 30, average reward: 702.7999877929688
Episode: 40, average reward: 400.1000061035156
0
dffff
Episode: 50, average reward: 760.0999755859375
Episode: 60, average reward: 570.9000244140625
0
dffff
Episode: 70, average reward: 702.2000122070312
Episode: 80, average reward: 502.70001220703125
0
dffff
Episode: 90, average reward: 563.5
Episode: 100, average reward: 610.9000244140625
Episode: 110, average reward: 678.4000244140625
0
dffff
Episode: 120, average reward: 488.3999938964844
Episode: 130, average reward: 598.7999877929688
0
dffff
Episode: 140, average reward: 599.0999755859375
Episode: 150, average reward: 557.5999755859375
0
dffff
Episode: 160, average reward: 675.7999877929688
Episode: 170, average reward: 806.0999755859375
0
dffff
Episode: 180, average reward: 792.2000122070312
Episode: 190, average reward: 720.2000122070312
0
dffff
Episode: 200, average reward: 657.90002441

KeyboardInterrupt: ignored

In [ ]:
import pickle 
path="/content/drive/MyDrive/my_Mario/PPOResults"
with open(os.path.join(path,"total_rewards.pkl"), "wb") as f:
    pickle.dump(solver.all_episode_rewards[:1000], f)
with open(os.path.join(path,"total_steps.pkl"), "wb") as f:
    pickle.dump(solver.steps[:1000], f)
with open(os.path.join(path,"total_loss.pkl"), "wb") as f:
    pickle.dump(solver.loss[:1000], f)
with open(os.path.join(path,"total_time.pkl"), "wb") as f:
    pickle.dump(solver.time[:1000], f)

In [ ]:
with open(os.path.join(path,"total_rewards.pkl"), 'rb') as f:
    total_rewards = pickle.load(f)
with open(os.path.join(path,"total_steps.pkl"), "rb") as f:
    total_steps = pickle.load(f)
with open(os.path.join(path,"total_loss.pkl"), "rb") as f:
    total_loss = pickle.load(f)
with open(os.path.join(path,"total_time.pkl"), "rb") as f:
    total_time = pickle.load(f)